# TASK 2: DATA CLEANING

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("TASK 2: LÀM SẠCH DỮ LIỆU ")

TASK 2: LÀM SẠCH DỮ LIỆU 


## LOAD DỮ LIỆU

In [2]:
print("\nLoading data...")
movies = pd.read_csv('../data/raw/movies_with_features.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')
users = pd.read_csv('../data/raw/users_with_features.csv')

print(f"Movies:  {len(movies):,} rows, {len(movies.columns)} columns")
print(f"Ratings: {len(ratings):,} rows")
print(f"Users:   {len(users):,} rows")

# Backup sizes
original_movies = len(movies)
original_ratings = len(ratings)
original_users = len(users)


Loading data...
Movies:  3,883 rows, 11 columns
Ratings: 1,000,209 rows
Users:   6,040 rows


## XỬ LÝ MISSING VALUES

In [3]:
print("XỬ LÝ MISSING VALUES")

print("\nMovies missing values:")
movies_null = movies.isnull().sum()
if movies_null.sum() > 0:
    print(movies_null[movies_null > 0])
else:
    print("Không có missing values")

print("\nRatings missing values:")
ratings_null = ratings.isnull().sum()
if ratings_null.sum() > 0:
    print(ratings_null[ratings_null > 0])
else:
    print("Không có missing values")

print("\nUsers missing values:")
users_null = users.isnull().sum()
if users_null.sum() > 0:
    print(users_null[users_null > 0])
else:
    print("Không có missing values")

print("\nXử lý missing values...")

# Movies: Fill missing values
if 'year' in movies.columns:
    year_missing = movies['year'].isnull().sum()
    if year_missing > 0:
        median_year = movies['year'].median()
        movies['year'].fillna(median_year, inplace=True)
        print(f"Filled {year_missing} missing 'year' with median: {median_year}")

if 'rating_avg' in movies.columns:
    movies['rating_avg'].fillna(0, inplace=True)
    movies['rating_count'].fillna(0, inplace=True)
    movies['rating_std'].fillna(0, inplace=True)
    print("Filled rating stats with 0 (no ratings yet)")

if 'decade' in movies.columns:
    decade_missing = movies['decade'].isnull().sum()
    if decade_missing > 0:
        median_year = movies['year'].median()
        movies['decade'].fillna((median_year // 10) * 10, inplace=True)
        print(f"Filled {decade_missing} missing 'decade'")

# Ratings: Drop rows với missing values (nếu có)
ratings_before = len(ratings)
ratings.dropna(inplace=True)
dropped = ratings_before - len(ratings)
if dropped > 0:
    print(f"Dropped {dropped} rows with missing values in ratings")
else:
    print("No missing values to drop in ratings")

# Users: Drop rows với missing values (nếu có)
users_before = len(users)
users.dropna(inplace=True)
dropped = users_before - len(users)
if dropped > 0:
    print(f"Dropped {dropped} rows with missing values in users")
else:
    print("No missing values to drop in users")

XỬ LÝ MISSING VALUES

Movies missing values:
rating_avg             177
rating_count           177
rating_std             291
popularity_category    177
dtype: int64

Ratings missing values:
Không có missing values

Users missing values:
Không có missing values

Xử lý missing values...
Filled rating stats with 0 (no ratings yet)
No missing values to drop in ratings
No missing values to drop in users


## LOẠI BỎ DUPLICATES

In [4]:
print("LOẠI BỎ DUPLICATES")

print("\nKiểm tra duplicates...")
movies_dup = movies['movieId'].duplicated().sum()
ratings_dup = ratings.duplicated(subset=['userId', 'movieId']).sum()
users_dup = users['userId'].duplicated().sum()

print(f"Movies duplicates (movieId): {movies_dup}")
print(f"Ratings duplicates (userId + movieId): {ratings_dup}")
print(f"Users duplicates (userId): {users_dup}")

print("\nXóa duplicates...")

# Movies: Remove duplicate movieId (giữ row đầu tiên)
movies_before = len(movies)
movies.drop_duplicates(subset=['movieId'], keep='first', inplace=True)
removed = movies_before - len(movies)
print(f"Movies: Removed {removed} duplicates")

# Ratings: Remove duplicate (userId, movieId) - giữ rating mới nhất
ratings_before = len(ratings)
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')
removed = ratings_before - len(ratings)
print(f"Ratings: Removed {removed} duplicates (kept latest)")

# Users: Remove duplicate userId
users_before = len(users)
users.drop_duplicates(subset=['userId'], keep='first', inplace=True)
removed = users_before - len(users)
print(f"Users: Removed {removed} duplicates")

LOẠI BỎ DUPLICATES

Kiểm tra duplicates...
Movies duplicates (movieId): 0
Ratings duplicates (userId + movieId): 0
Users duplicates (userId): 0

Xóa duplicates...
Movies: Removed 0 duplicates
Ratings: Removed 0 duplicates (kept latest)
Users: Removed 0 duplicates


## XỬ LÝ OUTLIERS

In [5]:
print("XỬ LÝ OUTLIERS")

print("\nPhân tích outliers...")

# Movies: Loại movies có quá ít ratings
min_ratings_per_movie = 5
movies_with_few_ratings = movies[movies['rating_count'] < min_ratings_per_movie]
print(f"\nMovies với < {min_ratings_per_movie} ratings: {len(movies_with_few_ratings):,} / {len(movies):,}")

# Users: Loại users có quá ít ratings
min_ratings_per_user = 3
user_rating_counts = ratings['userId'].value_counts()
users_with_few_ratings = user_rating_counts[user_rating_counts < min_ratings_per_user]
print(f"Users với < {min_ratings_per_user} ratings: {len(users_with_few_ratings):,} / {len(user_rating_counts):,}")

# Ratings: Kiểm tra ratings bất thường
print(f"\nRating range: {ratings['rating'].min()} - {ratings['rating'].max()}")
print(f"Valid range: 1-5")
invalid_ratings = ratings[(ratings['rating'] < 1) | (ratings['rating'] > 5)]
print(f"Invalid ratings: {len(invalid_ratings)}")

print("\nLoại bỏ outliers...")

# Loại movies có ít ratings
movies_before = len(movies)
valid_movies = movies[movies['rating_count'] >= min_ratings_per_movie]['movieId']
movies = movies[movies['movieId'].isin(valid_movies)]
print(f"Removed {movies_before - len(movies):,} movies with < {min_ratings_per_movie} ratings")

# Loại users có ít ratings
ratings_before = len(ratings)
valid_users = user_rating_counts[user_rating_counts >= min_ratings_per_user].index
ratings = ratings[ratings['userId'].isin(valid_users)]
users = users[users['userId'].isin(valid_users)]
print(f"Removed {ratings_before - len(ratings):,} ratings from users with < {min_ratings_per_user} ratings")

# Loại ratings không hợp lệ (nếu có)
ratings_before = len(ratings)
ratings = ratings[(ratings['rating'] >= 1) & (ratings['rating'] <= 5)]
removed = ratings_before - len(ratings)
print(f"Removed {removed} invalid ratings")

# Chỉ giữ movies có trong ratings
valid_movie_ids = ratings['movieId'].unique()
movies_before = len(movies)
movies = movies[movies['movieId'].isin(valid_movie_ids)]
print(f"Removed {movies_before - len(movies):,} movies without valid ratings")

# RECALCULATE RATING STATS AFTER CLEANING
print("\nRecalculate rating stats sau khi cleaning...")
rating_stats = ratings.groupby('movieId').agg({
    'rating': ['mean', 'count', 'std']
}).reset_index()
rating_stats.columns = ['movieId', 'rating_avg', 'rating_count', 'rating_std']

# Drop old rating columns
movies = movies.drop(columns=['rating_avg', 'rating_count', 'rating_std'], errors='ignore')

# Merge new stats
movies = movies.merge(rating_stats, on='movieId', how='left')
movies['rating_avg'].fillna(0, inplace=True)
movies['rating_count'].fillna(0, inplace=True)
movies['rating_std'].fillna(0, inplace=True)

print(f"Recalculated rating stats for {len(movies):,} movies")
print(f"Avg ratings per movie: {movies['rating_count'].mean():.2f}")
print(f"Min ratings per movie: {movies['rating_count'].min():.0f}")
print(f"Max ratings per movie: {movies['rating_count'].max():.0f}")

# RECALCULATE POPULARITY_CATEGORY 
print("\nRecalculate popularity_category sau khi cleaning...")

# Drop old popularity_category (vì đã outdated)
if 'popularity_category' in movies.columns:
    movies = movies.drop(columns=['popularity_category'])
    print("Dropped old popularity_category")

# Recalculate với rating_count MỚI
movies['popularity_category'] = pd.cut(
    movies['rating_count'],
    bins=[0, 10, 50, 200, float('inf')],
    labels=['Niche', 'Moderate', 'Popular', 'Blockbuster']
)

# Check NaN và fill (nếu có movies với rating_count = 0)
null_pop = movies['popularity_category'].isnull().sum()
if null_pop > 0:
    print(f"{null_pop} movies có popularity_category = NaN (rating_count = 0)")
    movies['popularity_category'].fillna('Niche', inplace=True)
    print(f"Filled NaN with 'Niche'")
else:
    print(f"No NaN in popularity_category")

print(f"Recalculated popularity_category for {len(movies):,} movies")
print(f"\nDistribution:")
pop_dist = movies['popularity_category'].value_counts().sort_index()
for cat, count in pop_dist.items():
    print(f"{cat:<12}: {count:>5,} movies ({count/len(movies)*100:>5.2f}%)")


XỬ LÝ OUTLIERS

Phân tích outliers...

Movies với < 5 ratings: 467 / 3,883
Users với < 3 ratings: 0 / 6,040

Rating range: 1 - 5
Valid range: 1-5
Invalid ratings: 0

Loại bỏ outliers...
Removed 467 movies with < 5 ratings
Removed 0 ratings from users with < 3 ratings
Removed 0 invalid ratings
Removed 0 movies without valid ratings

Recalculate rating stats sau khi cleaning...
Recalculated rating stats for 3,416 movies
Avg ratings per movie: 292.63
Min ratings per movie: 5
Max ratings per movie: 3428

Recalculate popularity_category sau khi cleaning...
Dropped old popularity_category
No NaN in popularity_category
Recalculated popularity_category for 3,416 movies

Distribution:
Niche       :   183 movies ( 5.36%)
Moderate    :   734 movies (21.49%)
Popular     : 1,079 movies (31.59%)
Blockbuster : 1,420 movies (41.57%)


## CHUẨN HÓA DỮ LIỆU

In [6]:
print("CHUẨN HÓA DỮ LIỆU")

print("\nNormalize ratings...")
# Giữ nguyên ratings [1, 5] (không normalize) vì SVD và NCF cần giá trị này
print("Keeping ratings in original range [1, 5] for SVD/NCF compatibility")

print("\nExtract và clean year...")
# Đã làm ở Task 1, kiểm tra lại
if 'year' in movies.columns:
    movies['year'] = movies['year'].astype('Int64')  # Int64 hỗ trợ NaN
    print(f"Year range: {movies['year'].min()} - {movies['year'].max()}")

print("\nClean text...")
# Clean title (nếu chưa có)
if 'title_clean' not in movies.columns:
    movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()
    print("✓ Created title_clean")

# Lowercase và remove special chars cho genres
movies['genres_clean'] = movies['genres'].str.lower().str.replace('[^a-z0-9|]', '', regex=True)
print("Cleaned text: title_clean, genres_clean")

print("\nChuẩn hóa ID...")
# Reset index để đảm bảo continuous
movies.reset_index(drop=True, inplace=True)
ratings.reset_index(drop=True, inplace=True)
users.reset_index(drop=True, inplace=True)
print("Reset all indices")

CHUẨN HÓA DỮ LIỆU

Normalize ratings...
Keeping ratings in original range [1, 5] for SVD/NCF compatibility

Extract và clean year...
Year range: 1919 - 2000

Clean text...
Cleaned text: title_clean, genres_clean

Chuẩn hóa ID...
Reset all indices


## VECTOR HÓA (TF-IDF) - CONTENT-BASED

In [7]:
print("TÁC VỤ 5: VECTOR HÓA (TF-IDF) - CONTENT-BASED")

from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import os

print("\nChuẩn bị text corpus...")
# Combine genres + title_clean thành text
movies['text_features'] = movies['genres'].fillna('') + ' ' + movies['title_clean'].fillna('')
movies['text_features'] = movies['text_features'].str.lower().str.strip()
print(f"Created text_features from genres + title")
print(f"Sample: '{movies['text_features'].iloc[0]}'")

print("\nTF-IDF Vectorization...")
tfidf = TfidfVectorizer(
    max_features=5000,      # Giới hạn features để tiết kiệm memory
    stop_words='english',   # Loại bỏ stop words
    ngram_range=(1, 2),     # Unigrams + bigrams
    min_df=2,               # Từ phải xuất hiện ít nhất 2 lần
    max_df=0.8              # Loại từ xuất hiện quá nhiều (>80% docs)
)

tfidf_matrix = tfidf.fit_transform(movies['text_features'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Movies: {tfidf_matrix.shape[0]:,}")
print(f"Features (words): {tfidf_matrix.shape[1]:,}")
print(f"Non-zero elements: {tfidf_matrix.nnz:,}")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

print("\nLưu TF-IDF artifacts...")
os.makedirs('../models', exist_ok=True)

# Save TF-IDF matrix
with open('../models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
print("Saved: models/tfidf_matrix.pkl")

# Save TF-IDF vectorizer
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Saved: models/tfidf_vectorizer.pkl")

# Save movie_id to index mapping
movie_indices = pd.Series(movies.index, index=movies['movieId']).to_dict()
with open('../models/movie_indices.pkl', 'wb') as f:
    pickle.dump(movie_indices, f)
print("Saved: models/movie_indices.pkl")


TÁC VỤ 5: VECTOR HÓA (TF-IDF) - CONTENT-BASED

Chuẩn bị text corpus...
Created text_features from genres + title
Sample: 'animation|children's|comedy toy story'

TF-IDF Vectorization...
TF-IDF Matrix shape: (3416, 1671)
Movies: 3,416
Features (words): 1,671
Non-zero elements: 14,487
Sparsity: 99.75%

Lưu TF-IDF artifacts...
Saved: models/tfidf_matrix.pkl
Saved: models/tfidf_vectorizer.pkl
Saved: models/movie_indices.pkl


## TỔNG KẾT & LƯU FILE

In [8]:
print("TỔNG KẾT")

print("\nData Cleaning Summary:")
print(f"{'Metric':<30} {'Before':<15} {'After':<15} {'Removed':<15} {'% Removed':<12}")
print("-" * 90)
print(f"{'Movies':<30} {original_movies:<15,} {len(movies):<15,} {original_movies - len(movies):<15,} {(original_movies - len(movies))/original_movies*100:<11.2f}%")
print(f"{'Ratings':<30} {original_ratings:<15,} {len(ratings):<15,} {original_ratings - len(ratings):<15,} {(original_ratings - len(ratings))/original_ratings*100:<11.2f}%")
print(f"{'Users':<30} {original_users:<15,} {len(users):<15,} {original_users - len(users):<15,} {(original_users - len(users))/original_users*100:<11.2f}%")

print("\nData Quality Metrics:")
print(f"Average ratings per movie: {ratings.groupby('movieId').size().mean():.2f}")
print(f"Average ratings per user:  {ratings.groupby('userId').size().mean():.2f}")
print(f"Min ratings per movie:     {ratings.groupby('movieId').size().min()}")
print(f"Max ratings per movie:     {ratings.groupby('movieId').size().max()}")
print(f"Min ratings per user:      {ratings.groupby('userId').size().min()}")
print(f"Max ratings per user:      {ratings.groupby('userId').size().max()}")

print("\nRating distribution:")
rating_dist = ratings['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    print(f"  {rating} stars: {count:,} ({count/len(ratings)*100:.2f}%)")

print("\nSaving cleaned data...")
os.makedirs('../data/cleaned', exist_ok=True)

movies.to_csv('../data/cleaned/movies_cleaned.csv', index=False, encoding='utf-8')
print("Saved: data/cleaned/movies_cleaned.csv")

ratings.to_csv('../data/cleaned/ratings_cleaned.csv', index=False, encoding='utf-8')
print("Saved: data/cleaned/ratings_cleaned.csv")

users.to_csv('../data/cleaned/users_cleaned.csv', index=False, encoding='utf-8')
print("Saved: data/cleaned/users_cleaned.csv")

# Save cleaning report
cleaning_report = {
    'original_movies': original_movies,
    'original_ratings': original_ratings,
    'original_users': original_users,
    'cleaned_movies': len(movies),
    'cleaned_ratings': len(ratings),
    'cleaned_users': len(users),
    'movies_removed': original_movies - len(movies),
    'ratings_removed': original_ratings - len(ratings),
    'users_removed': original_users - len(users),
    'movies_removed_pct': (original_movies - len(movies))/original_movies*100,
    'ratings_removed_pct': (original_ratings - len(ratings))/original_ratings*100,
    'users_removed_pct': (original_users - len(users))/original_users*100,
    'avg_ratings_per_movie': float(ratings.groupby('movieId').size().mean()),
    'avg_ratings_per_user': float(ratings.groupby('userId').size().mean()),
    'min_ratings_per_movie': int(ratings.groupby('movieId').size().min()),
    'max_ratings_per_movie': int(ratings.groupby('movieId').size().max()),
    'min_ratings_per_user': int(ratings.groupby('userId').size().min()),
    'max_ratings_per_user': int(ratings.groupby('userId').size().max()),
    'tfidf_features': tfidf_matrix.shape[1],
    'tfidf_sparsity': float((1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100)
}

report_df = pd.DataFrame([cleaning_report])
report_df.to_csv('../data/cleaned/cleaning_report.csv', index=False)
print("Saved: data/cleaned/cleaning_report.csv")
print("TASK 2 HOÀN THÀNH!")


TỔNG KẾT

Data Cleaning Summary:
Metric                         Before          After           Removed         % Removed   
------------------------------------------------------------------------------------------
Movies                         3,883           3,416           467             12.03      %
Ratings                        1,000,209       1,000,209       0               0.00       %
Users                          6,040           6,040           0               0.00       %

Data Quality Metrics:
Average ratings per movie: 269.89
Average ratings per user:  165.60
Min ratings per movie:     1
Max ratings per movie:     3428
Min ratings per user:      20
Max ratings per user:      2314

Rating distribution:
  1 stars: 56,174 (5.62%)
  2 stars: 107,557 (10.75%)
  3 stars: 261,197 (26.11%)
  4 stars: 348,971 (34.89%)
  5 stars: 226,310 (22.63%)

Saving cleaned data...
Saved: data/cleaned/movies_cleaned.csv
Saved: data/cleaned/ratings_cleaned.csv
Saved: data/cleaned/users_clean